In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

In [2]:
DATA_PATH = "raw_car_dataset.csv"
df = pd.read_csv(DATA_PATH)

In [3]:
# initial snapshoot head
print("\n====== initial head =====")
print(df.head())


====== initial head =====
    Price  Odometer_km  Doors  Accidents Location  Year
0  $1,500     137830.0    4.0          1     City  1998
1  4171.0          NaN    4.0          0    Rural  2016
2  $5,331     107302.0    4.0          0   Suburb  2014
3  1500.0     141838.0    4.0          1   Suburb  1999
4  1500.0          NaN    3.0          0     City  2022


In [4]:
# intial info 
print("\n ===== intial info ====")
print(df.info())


 ===== intial info ====
<class 'pandas.DataFrame'>
RangeIndex: 145 entries, 0 to 144
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Price        145 non-null    str    
 1   Odometer_km  138 non-null    float64
 2   Doors        138 non-null    float64
 3   Accidents    145 non-null    int64  
 4   Location     140 non-null    str    
 5   Year         145 non-null    int64  
dtypes: float64(2), int64(2), str(2)
memory usage: 6.9 KB
None


In [5]:
# initial misssin values

print("\n ===== initial missing values ====")
print(df.isnull().sum())


 ===== initial missing values ====
Price          0
Odometer_km    7
Doors          7
Accidents      0
Location       5
Year           0
dtype: int64


In [6]:
# fix categorial like location
df["Price"] = df["Price"].replace(r"[\$,]", "", regex=True).astype(float)

In [7]:
# Fix Category Errors before Imputation
df["Location"] = df["Location"].replace({"Subrb" : "Suburb" , "??" : pd.NA})

In [8]:
# imputing missing values

df["Location"] = df["Location"].fillna(df["Location"].mode()[0])
df["Doors"] = df["Doors"].fillna(df["Doors"].mode()[0])
df["Odometer_km"] = df["Odometer_km"].fillna(df["Odometer_km"].median())

In [9]:
print("\n ===== initial after fixing and imputing values ====")
print(df.isnull().sum())


 ===== initial after fixing and imputing values ====
Price          0
Odometer_km    0
Doors          0
Accidents      0
Location       0
Year           0
dtype: int64


In [10]:
# removing duplicate 
before_duplicte_remove = df.shape

df = df.drop_duplicates()

after_duplicate_remove = df.shape

print(f"this is before {before_duplicte_remove} \n  this is after {after_duplicate_remove}" )

this is before (145, 6) 
  this is after (140, 6)


In [11]:
# outlier this is so hard macalin wllhi 
def quantile_xisaabin(series, k=1.5):

    Q1, Q3 = series.quantile([0.25,0.75])
    
    IQR = Q3-Q1
    
    lower_quan = Q1 - (k*IQR)
    upper_quan = Q3 + (k*IQR)

    return lower_quan, upper_quan

lower_price , upper_price = quantile_xisaabin(df["Price"])
lower_Odometer_km , upper_Odometer_km = quantile_xisaabin(df["Odometer_km"])


df["Price"] = df["Price"].clip(lower=lower_price, upper=upper_price)
df["Odometer_km"] = df["Odometer_km"].clip(lower= lower_Odometer_km, upper=upper_Odometer_km)



In [12]:
# one hot encoding
df = pd.get_dummies(df, columns=["Location"], drop_first = False , dtype="int")


In [13]:
print(df.head())

    Price  Odometer_km  Doors  Accidents  Year  Location_City  Location_Rural  \
0  1500.0     137830.0    4.0          1  1998              1               0   
1  4171.0     128548.0    4.0          0  2016              0               1   
2  5331.0     107302.0    4.0          0  2014              0               0   
3  1500.0     141838.0    4.0          1  1999              0               0   
4  1500.0     128548.0    3.0          0  2022              1               0   

   Location_Suburb  
0                0  
1                0  
2                1  
3                1  
4                0  


In [14]:
# Feature Engineering  ustaad omeroow engineering baan iska yar baadh baadhi igu qaado hada wax qaado

CURRENT_YEAR = 2026

df["Car_age"] = CURRENT_YEAR - df["Year"] 
df["Km_per_year"] = df["Odometer_km"] / df["Car_age"]
df["is_Urbun"] = df["Location_City"] + df["Location_Suburb"]

# kan anaa keenay i aamin

df["is_Accident"] = (df["Accidents"] > 0).astype(int)


In [15]:
# aan arko in lagu daray feuture engineering ka iyo in kal 
print(df.head())

    Price  Odometer_km  Doors  Accidents  Year  Location_City  Location_Rural  \
0  1500.0     137830.0    4.0          1  1998              1               0   
1  4171.0     128548.0    4.0          0  2016              0               1   
2  5331.0     107302.0    4.0          0  2014              0               0   
3  1500.0     141838.0    4.0          1  1999              0               0   
4  1500.0     128548.0    3.0          0  2022              1               0   

   Location_Suburb  Car_age   Km_per_year  is_Urbun  is_Accident  
0                0       28   4922.500000         1            1  
1                0       10  12854.800000         0            0  
2                1       12   8941.833333         1            0  
3                1       27   5253.259259         1            1  
4                0        4  32137.000000         1            0  


In [16]:
# scaling on of the best waan ku raaxeesanooyaa 
dont_Scale = ["Price"]
numeric_cols = df.select_dtypes(include=["int64", "float64"]).columns.to_list()
exclude_Features = [ c for c in df.columns if c.startswith("Location_")] + ["is_Urbun"] + ["is_Accident"]
num_features_to_scale = [c for c in numeric_cols if c not in dont_Scale and c not in exclude_Features]

scaler = StandardScaler()
df[num_features_to_scale] = scaler.fit_transform( df[num_features_to_scale])





In [17]:
# final snap shoot oo macaan waliba 
print("\n ==== final head ====")
print(df.head())


 ==== final head ====
    Price  Odometer_km     Doors  Accidents      Year  Location_City  \
0  1500.0     0.128390  0.254091   0.316968 -1.686714              1   
1  4171.0    -0.044709  0.254091  -0.820867  0.794617              0   
2  5331.0    -0.440923  0.254091  -0.820867  0.518913              0   
3  1500.0     0.203135  0.254091   0.316968 -1.548862              0   
4  1500.0    -0.044709 -0.931668  -0.820867  1.621727              1   

   Location_Rural  Location_Suburb   Car_age  Km_per_year  is_Urbun  \
0               0                0  1.686714    -0.615631         1   
1               1                0 -0.794617     0.070446         0   
2               0                1 -0.518913    -0.267993         1   
3               0                1  1.548862    -0.587024         1   
4               0                0 -1.621727     1.738196         1   

   is_Accident  
0            1  
1            0  
2            0  
3            1  
4            0  


In [18]:
# final snapshot info
print("\n final info malap ah")
print(df.info())


 final info malap ah
<class 'pandas.DataFrame'>
RangeIndex: 140 entries, 0 to 139
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Price            140 non-null    float64
 1   Odometer_km      140 non-null    float64
 2   Doors            140 non-null    float64
 3   Accidents        140 non-null    float64
 4   Year             140 non-null    float64
 5   Location_City    140 non-null    int64  
 6   Location_Rural   140 non-null    int64  
 7   Location_Suburb  140 non-null    int64  
 8   Car_age          140 non-null    float64
 9   Km_per_year      140 non-null    float64
 10  is_Urbun         140 non-null    int64  
 11  is_Accident      140 non-null    int64  
dtypes: float64(7), int64(5)
memory usage: 13.3 KB
None


In [20]:
# final missing value
print("\n ==== final missin value malab ah ====")
print(df.isnull().sum())


 ==== final missin value malab ah ====
Price              0
Odometer_km        0
Doors              0
Accidents          0
Year               0
Location_City      0
Location_Rural     0
Location_Suburb    0
Car_age            0
Km_per_year        0
is_Urbun           0
is_Accident        0
dtype: int64


In [26]:
# save data 
df.to_csv("clean_car_dataset.csv", index=False)